# Customer Retention & Churn-Risk Analysis

**Quick-Commerce Analytics | Business Question 7: Which customers are likely to stop ordering?**

This notebook examines customer inactivity, evaluates a logistic regression model for predicting **no completed purchase within a three-month observation window**, and scores customers for potential retention or reactivation outreach.

**Tools:** Python, pandas, scikit-learn. **Data source:** Customer-level extracts prepared in PostgreSQL from the 2025 synthetic quick-commerce dataset.

### Input files

Place these three SQL exports alongside this notebook before running it:

- `customer_purchase_history.csv` — descriptive customer history through 31 December 2025
- `customer_churn_training.csv` — historical features through 30 September 2025 and October–December non-return labels
- `customer_churn_scoring.csv` — nine-month features through 31 December 2025 for future risk scoring

The notebook runs top to bottom. The original analysis code and its saved outputs have been preserved.

## 1. Descriptive Customer Inactivity

Inspect all registered customers, including customers who never completed an order. Missing last-order dates and recency values are expected for customers without qualifying orders.

In [1]:
import pandas as pd

df = pd.read_csv("customer_purchase_history.csv")

print(df.shape)
print(df.info())
print(df.isna().sum())

df.head(10)

(8500, 5)
<class 'pandas.DataFrame'>
RangeIndex: 8500 entries, 0 to 8499
Data columns (total 5 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   customer_id             8500 non-null   str    
 1   last_order_date         8359 non-null   str    
 2   total_completed_orders  8500 non-null   int64  
 3   total_spend_inr         8500 non-null   float64
 4   days_since_last_order   8359 non-null   float64
dtypes: float64(2), int64(1), str(2)
memory usage: 464.6 KB
None
customer_id                 0
last_order_date           141
total_completed_orders      0
total_spend_inr             0
days_since_last_order     141
dtype: int64


,customer_id,last_order_date,total_completed_orders,total_spend_inr,days_since_last_order
0,C05435,2025-01-03,1,2794.82,362.0
1,C02562,2025-01-03,1,1513.23,362.0
2,C03459,2025-01-06,1,1596.42,359.0
3,C06153,2025-01-08,1,1900.05,357.0
4,C04603,2025-01-09,1,2280.15,356.0
5,C04980,2025-01-10,3,3347.64,355.0
6,C00281,2025-01-19,1,895.18,346.0
7,C04949,2025-01-20,1,328.18,345.0
8,C07278,2025-01-20,2,1677.24,345.0
9,C03325,2025-01-20,1,944.51,345.0


### Inactivity groups

Categorize the number of days since each customer's most recent completed order as of 31 December 2025. These buckets describe **current inactivity**, not confirmed churn or future churn probability.

In [2]:
# Convert order dates to datetime
df["last_order_date"] = pd.to_datetime(df["last_order_date"])

# Categorize customers by inactivity
def inactivity_group(days):
    if pd.isna(days):
        return "Never purchased"
    elif days <= 30:
        return "0–30 days"
    elif days <= 60:
        return "31–60 days"
    elif days <= 90:
        return "61–90 days"
    else:
        return "Over 90 days"

df["inactivity_group"] = df["days_since_last_order"].apply(inactivity_group)

# Count customers in each group
group_order = [
    "0–30 days",
    "31–60 days",
    "61–90 days",
    "Over 90 days",
    "Never purchased"
]

df["inactivity_group"].value_counts().reindex(group_order)

inactivity_group
0–30 days          4649
31–60 days         1536
61–90 days          751
Over 90 days       1423
Never purchased     141
Name: count, dtype: int64

**Descriptive results:** 4,649 customers ordered within the last 30 days; 1,423 had been inactive for over 90 days; 141 had never completed an order. The full population is 8,500 customers.

## 2. Historical Churn Training Data

Use a historical cutoff to separate information available at prediction time from subsequent outcomes:

| Component | Period / definition |
|---|---|
| Features | Completed purchases before **1 October 2025** (2025 data: January–September) |
| Outcome | Completed purchases during **October–December 2025** |
| `churn_label = 1` | No completed order during the outcome period |
| `churn_label = 0` | At least one completed order during the outcome period |

Only customers with a completed purchase before the cutoff enter the historical training population. This is an operational **non-return** label, not proof that a customer has permanently churned.

In [3]:
churn_df = pd.read_csv("customer_churn_training.csv")

print(churn_df.shape)
print(churn_df["churn_label"].value_counts())
print(churn_df["churn_label"].value_counts(normalize=True) * 100)

churn_df.head()

(8035, 5)
churn_label
0    6636
1    1399
Name: count, dtype: int64
churn_label
0    82.588675
1    17.411325
Name: proportion, dtype: float64


,customer_id,recency_days,frequency,monetary_value_inr,churn_label
0,C00001,1,18,19157.64,0
1,C00002,3,23,24042.16,0
2,C00003,8,7,6668.16,1
3,C00004,15,13,12188.88,0
4,C00006,29,8,9388.31,1


**Class balance:** 8,035 previously purchasing customers; 1,399 (17.41%) did not return during October–December, while 6,636 (82.59%) returned. Because the non-return group is smaller, overall accuracy alone is not sufficient to judge performance.

## 3. Baseline Logistic Regression

Predict the historical non-return label using three **RFM** variables:

- **Recency:** days since last order at the historical cutoff
- **Frequency:** completed orders before the cutoff
- **Monetary value:** historical net revenue in INR

Use a stratified customer-level train/test split. Feature scaling occurs inside the training pipeline to avoid fitting the scaler on test data. The default decision threshold is 0.50.

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report, roc_auc_score

# Features and target
X = churn_df[
    ["recency_days", "frequency", "monetary_value_inr"]
]
y = churn_df["churn_label"]

# Stratified split preserves class proportions
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Scale features and train the classifier
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000)
)

model.fit(X_train, y_train)

# Evaluate on held-out customers
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 4))

              precision    recall  f1-score   support

           0       0.84      0.99      0.91      1327
           1       0.56      0.08      0.14       280

    accuracy                           0.83      1607
   macro avg       0.70      0.53      0.52      1607
weighted avg       0.79      0.83      0.77      1607

ROC-AUC: 0.7586


**Baseline interpretation:** ROC-AUC was **0.7586**, indicating some ability to rank customers by non-return risk. However, at the 0.50 threshold, the model's recall for non-returning customers was only **8%**, despite **83%** overall accuracy. Predicting that everyone returns would already achieve approximately **82.6%** accuracy.

## 4. Select a Decision Threshold Using Validation Data

Compare thresholds on a **validation subset of the training data**, rather than choosing a threshold from the held-out test results. The business objective is to identify more potential non-returning customers while accepting that lower thresholds increase unnecessary outreach.

In [6]:
from sklearn.base import clone
from sklearn.metrics import precision_score, recall_score, f1_score

# Create a validation set from the existing training data
X_fit, X_val, y_fit, y_val = train_test_split(
    X_train, y_train,
    test_size=0.25,
    random_state=42,
    stratify=y_train
)

# Train a separate model for threshold evaluation
val_model = clone(model)
val_model.fit(X_fit, y_fit)

val_prob = val_model.predict_proba(X_val)[:, 1]

# Compare candidate decision thresholds
for threshold in [0.2, 0.3, 0.4, 0.5]:
    val_pred = (val_prob >= threshold).astype(int)

    print(
        f"Threshold: {threshold:.1f} | "
        f"Precision: {precision_score(y_val, val_pred, zero_division=0):.3f} | "
        f"Recall: {recall_score(y_val, val_pred):.3f} | "
        f"F1: {f1_score(y_val, val_pred, zero_division=0):.3f} | "
        f"Flagged: {val_pred.sum()}"
    )

Threshold: 0.2 | Precision: 0.312 | Recall: 0.589 | F1: 0.408 | Flagged: 528
Threshold: 0.3 | Precision: 0.384 | Recall: 0.296 | F1: 0.335 | Flagged: 216
Threshold: 0.4 | Precision: 0.376 | Recall: 0.146 | F1: 0.211 | Flagged: 109
Threshold: 0.5 | Precision: 0.463 | Recall: 0.089 | F1: 0.150 | Flagged: 54


**Threshold choice:** Among the four tested thresholds (0.20, 0.30, 0.40, 0.50), **0.20** gave the highest validation F1-score (**0.408**) and recall (**58.9%**). Its lower precision (**31.2%**) makes it more suitable for inexpensive, targeted outreach than automatic large discounts.

## 5. Evaluate the Selected Threshold on Held-Out Customers

Apply the validation-selected **0.20** threshold once to the original held-out test predictions. This measures performance on customers whose outcomes were not used to choose the threshold.

In [7]:
from sklearn.metrics import classification_report, confusion_matrix

# Apply our selected threshold to the held-out test set
final_threshold = 0.20

y_pred_final = (y_prob >= final_threshold).astype(int)

print(classification_report(y_test, y_pred_final))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_final))

print("Customers flagged:", y_pred_final.sum())

              precision    recall  f1-score   support

           0       0.91      0.72      0.80      1327
           1       0.33      0.65      0.44       280

    accuracy                           0.71      1607
   macro avg       0.62      0.68      0.62      1607
weighted avg       0.81      0.71      0.74      1607

Confusion Matrix:
[[956 371]
 [ 99 181]]
Customers flagged: 552


**Held-out test results (threshold 0.20):**

| Measure | Result |
|---|---:|
| ROC-AUC | 0.7586 |
| Recall for non-return (`1`) | 64.6% (181 of 280) |
| Precision for non-return (`1`) | 32.8% (181 of 552 flagged) |
| F1 for non-return (`1`) | 0.44 |
| True positives / False positives | 181 / 371 |
| False negatives / True negatives | 99 / 956 |

The model captures more non-returning customers than at the 0.50 threshold, but many flagged customers would return without intervention.

## 6. Score Customers as of 31 December 2025

Score customers using **April–December 2025** purchase histories, matching the nine-month duration of the historical feature window. Only customers with a completed purchase in this scoring window are included; excluded customers need separate consideration.

The January–March 2026 outcomes are **not observed** in this dataset. Scores are estimates, not verified future outcomes.

In [8]:
scoring_df = pd.read_csv("customer_churn_scoring.csv")

print(scoring_df.shape)
print(scoring_df.info())
print(scoring_df.isna().sum())

scoring_df.head()

(8285, 4)
<class 'pandas.DataFrame'>
RangeIndex: 8285 entries, 0 to 8284
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   customer_id         8285 non-null   str    
 1   recency_days        8285 non-null   int64  
 2   frequency           8285 non-null   int64  
 3   monetary_value_inr  8285 non-null   float64
dtypes: float64(1), int64(2), str(1)
memory usage: 307.6 KB
None
customer_id           0
recency_days          0
frequency             0
monetary_value_inr    0
dtype: int64


,customer_id,recency_days,frequency,monetary_value_inr
0,C00001,45,17,17184.15
1,C00002,17,20,19295.22
2,C00003,100,2,2199.62
3,C00004,7,16,16946.77
4,C00005,11,6,6460.51


### Generate risk estimates

Apply the same trained logistic regression pipeline and feature order to the year-end customer data. Flag customers with predicted non-return probability of **at least 0.20**. The estimated probabilities have not been separately calibrated.

In [9]:
# Use the same three features and column order as training
features = [
    "recency_days",
    "frequency",
    "monetary_value_inr"
]

X_scoring = scoring_df[features]

# Predict probability of no purchase in the following three months
scoring_df["churn_probability"] = model.predict_proba(X_scoring)[:, 1]

# Apply the threshold selected during validation
scoring_df["at_risk"] = (
    scoring_df["churn_probability"] >= 0.20
).astype(int)

# Inspect results
print(scoring_df["at_risk"].value_counts())

print(
    scoring_df["churn_probability"].describe()
)

# View the 10 customers with the highest estimated risk
scoring_df.sort_values(
    "churn_probability",
    ascending=False
).head(10)

at_risk
0    5306
1    2979
Name: count, dtype: int64
count    8285.000000
mean        0.190487
std         0.144551
min         0.000179
25%         0.090367
50%         0.158275
75%         0.244713
max         0.758122
Name: churn_probability, dtype: float64


,customer_id,recency_days,frequency,monetary_value_inr,churn_probability,at_risk
4918,C05044,272,1,3230.89,0.758122,1
3474,C03562,274,1,2330.01,0.757454,1
207,C00215,274,1,1264.15,0.752884,1
4126,C04227,272,1,1969.93,0.752719,1
7643,C07840,274,1,937.35,0.751471,1
6242,C06408,272,1,1677.51,0.751455,1
5765,C05916,273,1,1120.84,0.750654,1
6145,C06308,273,1,771.72,0.749135,1
5460,C05599,273,1,703.84,0.748839,1
496,C00515,271,1,1226.85,0.747874,1


**Scoring results:** 8,285 customers scored; **2,979** flagged at the selected threshold and **5,306** not flagged. The median estimated non-return probability was **15.83%**. Customers with the highest scores typically had very long inactivity periods and only one completed purchase.

## 7. Distinguish Retention from Reactivation

Separate flagged customers according to their existing inactivity at year-end:

- **Retention target:** 90 days or less since last completed purchase
- **Reactivation target:** more than 90 days since last completed purchase

The 90-day boundary is a practical outreach distinction, **not** a validated definition of churn.

In [10]:
# Categorize flagged customers by their current inactivity
at_risk_df = scoring_df[scoring_df["at_risk"] == 1].copy()

at_risk_df["action_group"] = at_risk_df["recency_days"].apply(
    lambda days: "Retention target" if days <= 90
    else "Reactivation target"
)

print(at_risk_df["action_group"].value_counts())

print(
    at_risk_df.groupby("action_group")
    .agg(
        customers=("customer_id", "count"),
        avg_churn_probability=("churn_probability", "mean"),
        avg_recency_days=("recency_days", "mean")
    )
)

action_group
Retention target       1663
Reactivation target    1316
Name: count, dtype: int64
                     customers  avg_churn_probability  avg_recency_days
action_group                                                           
Reactivation target       1316               0.449768        151.553951
Retention target          1663               0.252032         50.287432


## 8. Key Findings and Business Recommendation

| Segment | Flagged customers | Mean estimated non-return probability | Mean days since last order |
|---|---:|---:|---:|
| Retention | **1,663** | **25.2%** | **50.3** |
| Reactivation | **1,316** | **45.0%** | **151.6** |

**Recommended use:** Prioritize modest-cost, targeted outreach for the retention group and design a separate reactivation approach for long-inactive customers. Use a comparison group before attributing any increase in repeat purchases to the intervention.

### Limitations

- **Single historical cutoff:** the customer-level test split evaluates generalization across customers in the same period, not performance across multiple future periods.
- **False positives:** non-return precision was 32.8% at the selected 0.20 threshold; blanket discounting could be costly.
- **Uncalibrated probabilities:** risk scores are model estimates and should not be interpreted as independently validated probabilities.
- **Limited inputs:** recency, frequency, and spending capture only part of customer behaviour.
- **Synthetic data:** findings illustrate the analytical workflow and should not be presented as measured real-world business outcomes.

**Conclusion:** This analysis provides a defensible initial method to rank existing customers by likelihood of three-month non-return and distinguish prevention from reactivation opportunities. Further modelling is not required for the scope of this portfolio project.